# Introduction

In this notebook, we are going to learn about debugging practices that allow us to thing critically about our code and resolve issues that arise when our code/algorithms aren't working properly. I am also going to briefly discuss some of the concerns associated with using generative AI for programming and debugging. This notebook will be split into general python debugging and deep learning debugging. You can expect there to be a pretty steep increase in difficulty from the first section to the next, so don't be afraid to ask as many questions as you need to.     

For the purposes of this notebook, we will divide our bugs into two categories  
- **code errors**- code that is improperly written such that it triggers an error. An example of this is a SyntaxError
- **logical errors**- code that executes without the compiler raising an error but doesn't accomplish the desired task because the code is logically flawed

## Caution regarding generative AI tools

 It is in your best interest to go through this tutorial on your own without the use of generative AI tools. AI is not a substitute for good debugging skills, and there are two main reasons why.

 One reason is that AI is error prone and may not give you the correct output, and you actually have to understand your code to some degree to recognize this. Instead of being in a situation where you are blindly feeding the error-ridden code back into the chat-bot over and over again expecting a working result in the end, it is better to understand the code and have the skillset to be able to work through the issue on your own. Even if you do want to use gen AI, with good debugging skills, you will at least know how to evaluate responses from AI and properly incorporate some of its suggestions into your code instead of placing blind faith in the AI response by just copying and pasting its output.

 The second major reason why over reliance on generative AI in coding is detrimental is due to the presence of logical errors. At least when you encounter code errors, you get an error message, so you know to paste it into a chat-bot to resolve the issue. With logical errors, you won't even know that something is wrong with your code unless you are looking at your code critically. Furthermore, it's much harder to evaluate code critically when you didn't even write it yourself, and it was generated by AI.

# Imports

In [ ]:
import scipy
import matplotlib.pyplot as plt
import os
import pickle
import torch
from torch.utils.data import TensorDataset, DataLoader, Subset

# Python debugging

## Code errors

### Problem 1: Basics
This code is supposed to add two arrays.

In [ ]:
import numpy as np
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print(a + b)

[5 7 9]


### Problem 2: Window Sliding
You have a list of 20 signals that are 7s in length, where each signal is collected at 250Hz. The code is supposed to randomly retrieve a 1s slice from each signal to use as part of a validation set.

In [ ]:
fs = 250
np.random.seed(42)
signal = [np.random.randn(fs * 7) for _ in range(20)]
val_signals = []
inds = np.random.randint(0, 6*fs, 20)

# don't change anything above this line
for i, ind in enumerate(inds):
  val_signals.append(signal[ind][i: i + fs])

IndexError: list index out of range

### Problem 3: Signal Filtering

You are performing a bandpass filter on a signal that isolates the 8Hz to 30Hz frequency band (relavent to BCI Motor Imagery Classification), but there is an error in the filter code. Here are the docs for the butterworth filter: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.butter.html

In [ ]:
# bandpass filters attenuate frequencies outside of a band (crit_freq)
import numpy as np
order = 4
crit_freq = [8, 30]
sampling_freq = 125

sig = np.random.randn(sampling_freq * 10)

b, a = scipy.signal.butter(order, crit_freq, btype = 'bandpass')
processed_signal = scipy.signal.filtfilt(b, a, sig, 1)

ValueError: Digital filter critical frequencies must be 0 < Wn < 1

## Logical Errors

### Problem 4: Max absolute normalization
This code is supposed to normalize the array such that all elements have an absolute value less than or equal to 1. The second print statement should return True

In [ ]:
import numpy as np

a = np.array([3, -10, -1, 5, -2, 3, -4, 1, 9, 5]) # don't modify this array
normalized = np.abs(a / a.max())
# don't modify lines below
print(normalized)
print(np.all(np.abs(normalized) <= 1)) # checks that all array elements have abs value <= 1

[0.33333333 1.11111111 0.11111111 0.55555556 0.22222222 0.33333333
 0.44444444 0.11111111 1.         0.55555556]
False


###Problem 5: Window Sliding and Train-Test-Split
This code is supposed to take a signal sampled at 250Hz for 7s, which is 1750 measurements in length, and use window sliding to divide it into chunks that are 1s long with a shift of 10 measurements. The signal is also going to be split into train and validation lists (75%, 25%). There are two major errors with this code.

Hint: Should data in the train set also be in the validation set?

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split as tts
fs = 250
signal = list(np.random.randn(fs * 7))
segments = []
i = 0
while i < len(signal):
    segments.append(signal[i:i+fs])
    i += fs

train_sigs, val_sigs = tts(segments)

(5, 2)

### Problem 6: Directory Traversal and Data Loading


In [ ]:
# This code sets up the problem. It should run without errors.
os.mkdir("dataset")
os.mkdir("dataset/train")
os.mkdir("dataset/test")

for i in range(10):
  arr = np.random.randn(100)
  with open(f"dataset/train/left_{i}.pkl", "wb") as f:
    pickle.dump(arr, f)
  arr = np.random.randn(100)
  with open(f"dataset/test/right_{i}.pkl", "wb") as f:
    pickle.dump(arr, f)

for i in range(10):
  arr = np.random.randn(100)
  with open(f"dataset/test/left_{i}.pkl", "wb") as f:
    pickle.dump(arr, f)
  arr = np.random.randn(100)
  with open(f"dataset/train/right_{i}.pkl", "wb") as f:
    pickle.dump(arr, f)

Here we are traversing a directory with train and test folders, each with data stored as pickle files that represent the left and right hand classes in our fictitious Motor Imagery dataset. We want to read the array from each file and store it in separate lists depending on whether it's from the training or test set. We also want to store a label in the corresponding list denoting whether it is a left or right hand motor imagery signal; however, this code doesn't work properly. It has 4-5 logical errors.

Tip: Use print statements to help you out. Based on the previous cell and examining the directory, how many signals should be in the train and test lists.

In [ ]:
train_sigs = []
train_labels = []
test_sigs = []
test_labels = []
class_to_idx = {"left": 0, "right": 1}
# don't modify anything above

root = "dataset"
for j in range(len(os.listdir(f'{root}/train'))):
  files = os.listdir(f'{root}/train')
  for i in range(len(files)):
      file_path = os.path.join(root, "train", files[0])
      with open(file_path, "rb") as file:
          data = pickle.load(file)
          train_sigs.append(data)  # Store the data
          train_labels.append(class_to_idx.get(files[i])) # store label

files = os.listdir(f'{root}/test')
for i in range(len(files)):
    file_path = os.path.join(root, "test", files[i])
    with open(file_path, "rb") as file:
        data = pickle.load(file)
        test_sigs.extend(data)  # Store the data
        test_labels.append(class_to_idx.get(files[i])) # store label

# Pytorch debugging

## Code errors

## Problem 7: fix the following matrix multiplciation to create a 4x3 matrix

In [ ]:
import torch

A = torch.randn(2, 3)
B = torch.randn(4, 2)


C = A @ B

RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x3 and 4x2)

### Problem 8: General CUDA and Torch Errors
You need to enable a GPU runtime and restart this session before doing this.

This code tries to train a simple linear model with no activations on a synthetic fahrenheit-celcius dataset with some gaussian noise. There are 4 errors in this code, so be patient, it might take some time. Use print statements as needed.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
device = 'cuda'

celcius = torch.randint(-100, 100, 10000) # should produce 10000 random ints on [-100, 100)
# don't modify the next three lines and don't modify the SimpleRegressor class
fahrenheit = 1.8 * celcius + 32 + torch.randn(10000)
dataset = TensorDataset(celcius.view(-1, 1), fahrenheit.view(-1, 1))
train_dl = DataLoader(dataset, batch_size=100, shuffle=True)

class SimpleRegressor(torch.nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.fc1 = torch.nn.Linear(input_size, hidden_size)
    self.fc2 = torch.nn.Linear(hidden_size, output_size)

  def forward(self, x):
    y = self.fc1(x)
    y = self.fc2(y)
    return y

model = SimpleRegressor(100, 10, 1).to(device)
# after you fix this code, try running the line below without momentum as an exercise
# to see how much slower training converges
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5, momentum=0.9)
criterion = torch.nn.MSELoss()
model.train()

for epoch in range(15):
  avg_loss = 0
  for celc, fahr in train_dl:
    optimizer.zero_grad()
    preds = model(celc)
    loss = criterion(preds, fahr)
    loss.backward()
    optimizer.step()
    avg_loss += loss.item()
  print(f"Epoch: {epoch + 1} Avg_loss: {avg_loss / len(train_dl):.2f}")


In [ ]:
import matplotlib.pyplot as plt
preds = model(celcius.unsqueeze(1).to(device).to(torch.float32))
plt.scatter(celcius, preds.cpu().detach(), s=20)
plt.scatter(celcius, fahrenheit.detach().numpy(), s=5) # you can see the blue background

### Problem 9: CNN shape issues

This CNN is being run on a batch of grayscale images. The batch size is 64, and each image is 28 * 28. There seem to be some matrix multiplication issues stopping the image batch from completing a forward pass. There are three errors to be fixed in this code.

In [ ]:
images = torch.randn((64, 28, 28)) # don't modify this line

class BasicCNN(torch.nn.Module):
  # don't modify the init method
  def __init__(self, in_channels, out_classes):
    super().__init__()
    self.conv = torch.nn.Conv2d(in_channels, 8, kernel_size=3, padding=1)
    self.bn = torch.nn.BatchNorm2d(8)
    self.relu = torch.nn.ReLU()
    self.pool = torch.nn.MaxPool2d(2, 2)

    self.flatten = torch.nn.Flatten()
    self.fc = torch.nn.Linear(32, out_classes)

  def forward(self, x):
    y = self.conv(x)
    y = self.bn(y)
    y = self.relu(y)
    y = self.pool(y)
    y = self.fc(y)
    return y

model = BasicCNN(1, 10)
pred = model(images.view(64, 28, 28))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (7168x14 and 32x10)

## Logical errors

### Problem 10: Miscalculating accuracy

- Try to fix the following block of code, which isn't accurately calculating the accuracy

In [ ]:
import torch

seed = 42
torch.manual_seed(seed)
preds = torch.rand(1000,2) # probability predictions of the model
y_true = torch.nn.functional.one_hot(torch.randint(0, 2, (1000,)), 2)
# don't modify anything above

correct = (preds == y_true).sum()

accuracy = correct / len(preds)

print("Accuracy:", accuracy.item())

"""
Hints:
  1. Are we getting the number of correct predictions?
  2. What is the accuracy formula?
"""
print()

Accuracy: 0.0



### Problem 11: K-fold Validation

In this problem, we are performing K-fold validation on the MNIST dataset to make sure our model is robust. K-fold validation involves spliting the data into k partitions, and in each iteration one partition is considered the validation/test set while the remaining data is the training set. In effect, the model is reinitialized, trained on a new training set, and validated on a different fold in each iteration. There are two main errors in this code that make it an invalid implementation of K-fold cross-validation.

In [ ]:
from torchvision import datasets, transforms
from sklearn.model_selection import KFold
device = 'cuda'
# Download MNIST dataset
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())
total_dataset, _ = torch.utils.data.random_split(train_dataset, (0.4, 0.6))
class CNN(torch.nn.Module):
  # don't modify the init method
  def __init__(self, in_channels, out_classes):
    super().__init__()
    self.conv = torch.nn.Conv2d(in_channels, 16, kernel_size=3, padding=1)
    self.bn = torch.nn.BatchNorm2d(16)
    self.relu = torch.nn.ReLU()
    self.pool = torch.nn.MaxPool2d(2, 2)

    self.flatten = torch.nn.Flatten()
    self.fc = torch.nn.Linear(3136, out_classes)

  def forward(self, x):
    y = self.conv(x)
    y = self.bn(y)
    y = self.relu(y)
    y = self.pool(y)
    y = self.flatten(y)
    y = self.fc(y)
    return y
# don't change anything above this

model = CNN(1, 10).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
kf = KFold(5)

for fold, (train_idx, val_idx) in enumerate(kf.split(total_dataset)):
  print(f"Fold {fold + 1}")
  train_ds = Subset(total_dataset, train_idx)
  val_ds = Subset(total_dataset, val_idx)
  train_dl = DataLoader(train_ds, batch_size=100, shuffle=True)
  val_dl = DataLoader(val_ds, batch_size=100, shuffle=True)
  for epoch in range(10):
    avg_loss = 0
    for images, labels in train_dl:
      optimizer.zero_grad()
      preds = model(images.to(device))
      loss = criterion(preds, labels.to(device))
      loss.backward()
      optimizer.step()
      avg_loss += loss.item()
    print(f"Epoch: {epoch + 1}\tAvg_loss: {avg_loss / len(train_dl):.2f}", end="\t")
    avg_val_loss = 0
    for images, labels in val_dl:
      preds = model(images.to(device))
      loss = criterion(preds, labels.to(device))
      optimizer.step()
      avg_val_loss += loss.item()
    print(f"Avg_val_loss: {avg_val_loss / len(val_dl):.2f}")

test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transforms.ToTensor())
test_dl = DataLoader(test_dataset, batch_size=1000, shuffle=True)
test_loss = 0.0
test_accuracy = 0.0
for images, labels in test_dl:
  preds = model(images.to(device))
  loss = criterion(preds, labels.to(device))
  test_loss += loss.item()
  test_accuracy += (preds.argmax(dim=-1) == labels.to(device)).float().mean()
print(f"Test loss: {test_loss/len(test_dl):.2f} \t Test Accuracy: {test_accuracy/len(test_dl):.2f}")

Fold 1
Epoch: 1	Avg_loss: 0.35	Avg_val_loss: 23.87
Epoch: 2	Avg_loss: 3.17	Avg_val_loss: 6.19
Epoch: 3	Avg_loss: 0.94	Avg_val_loss: 1.23
Epoch: 4	Avg_loss: 0.28	Avg_val_loss: 1.77
Epoch: 5	Avg_loss: 0.33	Avg_val_loss: 0.19
Epoch: 6	Avg_loss: 0.11	Avg_val_loss: 1.08
Epoch: 7	Avg_loss: 0.22	Avg_val_loss: 1.71
Epoch: 8	Avg_loss: 0.28	Avg_val_loss: 0.32
Epoch: 9	Avg_loss: 0.12	Avg_val_loss: 0.17
Epoch: 10	Avg_loss: 0.07	Avg_val_loss: 0.61
Fold 2
Epoch: 1	Avg_loss: 0.15	Avg_val_loss: 0.12
Epoch: 2	Avg_loss: 0.07	Avg_val_loss: 1.32
Epoch: 3	Avg_loss: 0.25	Avg_val_loss: 0.40
Epoch: 4	Avg_loss: 0.12	Avg_val_loss: 1.47
Epoch: 5	Avg_loss: 0.22	Avg_val_loss: 0.28
Epoch: 6	Avg_loss: 0.11	Avg_val_loss: 0.12
Epoch: 7	Avg_loss: 0.06	Avg_val_loss: 0.35
Epoch: 8	Avg_loss: 0.11	Avg_val_loss: 0.16
Epoch: 9	Avg_loss: 0.07	Avg_val_loss: 0.73
Epoch: 10	Avg_loss: 0.17	Avg_val_loss: 0.15
Fold 3
Epoch: 1	Avg_loss: 0.07	Avg_val_loss: 0.27
Epoch: 2	Avg_loss: 0.09	Avg_val_loss: 0.06
Epoch: 3	Avg_loss: 0.06	Avg_va